In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

orders = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

items = pd.read_csv(
    "../data/raw/olist_order_items_dataset.csv",
    parse_dates=["shipping_limit_date"]
)

payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

products = pd.read_csv("../data/raw/olist_products_dataset.csv")

translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

In [3]:
products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products.rename(
    columns={
        "product_category_name_english": "category"
    },
    inplace=True
)

products.drop(
    columns="product_category_name",
    inplace=True
)

products["category"] = products["category"].fillna("Unknown")

products = products.dropna(
    subset=[
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
)


In [4]:
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month
orders["purchase_month_name"] = orders["order_purchase_timestamp"].dt.month_name()
orders["purchase_quarter"] = orders["order_purchase_timestamp"].dt.quarter
orders["purchase_weekday"] = orders["order_purchase_timestamp"].dt.day_name()
orders["purchase_hour"] = orders["order_purchase_timestamp"].dt.hour

orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

orders["is_late"] = (
    orders["delivery_delay"] > 0
).astype(int)

products["product_volume_cm3"] = (
    products["product_length_cm"]
    * products["product_width_cm"]
    * products["product_height_cm"]
)

products["product_weight_kg"] = (
    products["product_weight_g"] / 1000
)

reviews["review_sentiment"] = np.select(
    [
        reviews["review_score"] >= 4,
        reviews["review_score"] == 3,
        reviews["review_score"] <= 2
    ],
    [
        "Positive",
        "Neutral",
        "Negative"
    ],
    default="Unknown"
)

In [5]:
payments_agg = (
    payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value","sum"),
        payment_installments=("payment_installments","max"),
        payment_type=("payment_type",lambda x: x.mode()[0])
    )
    .reset_index()
)

In [6]:
reviews_agg = (
    reviews
    .sort_values("review_creation_date")
    .drop_duplicates("order_id", keep="last")
)

In [7]:
master = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="left"
)

print(master.shape)

(99441, 21)


In [8]:
master = pd.merge(
    master,
    items,
    on="order_id",
    how="left"
)

print(master.shape)

(113425, 27)


In [9]:
master = pd.merge(
    master,
    products,
    on="product_id",
    how="left"
)

print(master.shape)

(113425, 37)


In [10]:
master = pd.merge(
    master,
    sellers,
    on="seller_id",
    how="left"
)

print(master.shape)

(113425, 40)


In [11]:
master = pd.merge(
    master,
    payments_agg,
    on="order_id",
    how="left"
)

print(master.shape)

(113425, 43)


In [12]:
master = pd.merge(
    master,
    reviews_agg[
        [
            "order_id",
            "review_score",
            "review_sentiment"
        ]
    ],
    on="order_id",
    how="left"
)

print(master.shape)

(113425, 45)


In [13]:
master["total_price"] = (
    master["price"] +
    master["freight_value"]
)

In [16]:
print(master.shape)
master.head()
master.info()
master.isnull().sum()

(113425, 46)
<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 46 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  str           
 1   customer_id                    113425 non-null  str           
 2   order_status                   113425 non-null  str           
 3   order_purchase_timestamp       113425 non-null  datetime64[us]
 4   order_approved_at              113264 non-null  datetime64[us]
 5   order_delivered_carrier_date   111457 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  113425 non-null  datetime64[us]
 8   purchase_year                  113425 non-null  int32         
 9   purchase_month                 113425 non-null  int32         
 10  purchase_month_name            113425 non-null  str           
 11

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 161
order_delivered_carrier_date     1968
order_delivered_customer_date    3229
order_estimated_delivery_date       0
purchase_year                       0
purchase_month                      0
purchase_month_name                 0
purchase_quarter                    0
purchase_weekday                    0
purchase_hour                       0
delivery_days                    3229
delivery_delay                   3229
is_late                             0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
order_item_id                     775
product_id                        775
seller_id                         775
shipping_limit_date               775
price                             775
freight_valu

In [ ]:
print("Duplicate Rows :", master.duplicated().sum())

Duplicate Rows : 0


In [20]:
missing = (
    master.isnull()
          .sum()
          .sort_values(ascending=False)
)

missing = missing[missing > 0]

missing

order_delivered_customer_date    3229
delivery_days                    3229
delivery_delay                   3229
product_photos_qty               2379
product_description_lenght       2379
product_name_lenght              2379
order_delivered_carrier_date     1968
review_score                      961
review_sentiment                  961
product_weight_g                  793
category                          793
product_volume_cm3                793
product_height_cm                 793
product_width_cm                  793
product_length_cm                 793
product_weight_kg                 793
seller_city                       775
seller_state                      775
price                             775
seller_zip_code_prefix            775
shipping_limit_date               775
seller_id                         775
total_price                       775
freight_value                     775
order_item_id                     775
product_id                        775
order_approv

In [21]:
print(master.shape)

(113425, 46)


In [22]:
print(master["order_id"].nunique())

99441


In [23]:
import os

os.makedirs("../data/processed", exist_ok=True)

master.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)

print("Master dataset saved successfully!")

Master dataset saved successfully!
